# Perspective + OCR Dual-CNN

End-to-end model that reads a credit card number from a photo taken at an angle, without
being given the card's location.

1. **PerspectiveNet** — predicts the 3×3 homography that straightens the card
2. **Spatial Transformer** — applies that homography with `grid_sample` (differentiable, so
   the OCR loss can backpropagate into the geometry)
3. **OCRNet** — reads 16 digits from the straightened crop

**Loss** = `alpha * MSE(matrix, in normalised space)` + `beta * CrossEntropy(digits)`

The interesting part is that the two tasks supervise each other: a bad homography produces a
crop the OCR head cannot read, so digit errors push the geometry to improve.

In [ ]:
import os
import json

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

## Geometry conventions

Getting these right is the whole ballgame, so they are stated explicitly.

- The ground-truth matrix `M` was produced by
  `cv2.getPerspectiveTransform(quad -> rectangle)` where **`quad` is in the original image's
  pixel coordinates** and the destination rectangle is `RECT_W x RECT_H` = 512x323.
- So `M` maps *original-resolution* pixels into `[0, 511] x [0, 322]`.
- The network sees images resized to 256x256. `grid_sample` works in normalised `[-1, 1]`
  coordinates, which are resolution independent — so we inverse-map in **original** pixel
  space and normalise by the **original** image size, then sample from the resized tensor.

Mixing these three coordinate spaces up is the easiest way to get a transformer that
silently returns black images.

In [ ]:
RECT_W, RECT_H = 512, 323   # destination rectangle the GT matrices map into
IMG_SIZE = (256, 256)       # what the network sees
CARD_SIZE = (64, 128)       # (H, W) of the straightened crop fed to the OCR head

## Dataset

In [ ]:
class PerspectiveOCRDataset(Dataset):
    """
    Each sample provides:
      - image      : (3, 256, 256) resized, normalised to [-1, 1]
      - matrix     : (3, 3) ground-truth homography, in ORIGINAL pixel coordinates
      - label      : (16,) digit indices
      - orig_size  : (2,) original (width, height), needed to interpret the matrix
    """

    def __init__(self, img_dir, matrix_dir, label_dir, img_size=IMG_SIZE, orig_sizes=None):
        self.img_dir, self.matrix_dir, self.label_dir = img_dir, matrix_dir, label_dir
        self.image_files = sorted(f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png')))
        if not self.image_files:
            raise FileNotFoundError(f"No images found in {img_dir}")

        # The matrices are expressed in ORIGINAL-resolution pixel coordinates. If the images
        # on disk have been pre-resized for speed, their size no longer tells us that, so the
        # original dimensions are read from orig_sizes.json instead.
        self.orig_sizes = None
        if orig_sizes and os.path.exists(orig_sizes):
            with open(orig_sizes) as f:
                self.orig_sizes = json.load(f)
            print(f"Using original sizes from {orig_sizes}")

        self.chars = "0123456789"
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.idx_to_char = {i: c for i, c in enumerate(self.chars)}

        self.transform = transforms.Compose([
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])

    def encode_label(self, text):
        clean = text.replace(' ', '').strip()
        return torch.tensor([self.char_to_idx[c] for c in clean if c in self.char_to_idx],
                            dtype=torch.long)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        stem = os.path.splitext(img_name)[0]

        image = Image.open(os.path.join(self.img_dir, img_name)).convert('RGB')
        if self.orig_sizes is not None:
            orig_size = torch.tensor(self.orig_sizes[stem], dtype=torch.float32)
        else:
            orig_size = torch.tensor(image.size, dtype=torch.float32)   # (W, H) before resizing
        image = self.transform(image)

        matrix = np.load(os.path.join(self.matrix_dir, f"{stem}.npy")).astype(np.float32)

        with open(os.path.join(self.label_dir, f"{stem}.txt")) as f:
            label = self.encode_label(f.read().strip())

        return {'image': image,
                'matrix': torch.from_numpy(matrix),
                'label': label,
                'orig_size': orig_size}

## Matrix normalisation

The nine elements of a homography live on wildly different scales — translation terms reach
±1400 while the projective terms sit around 1e-5. An unweighted MSE over the raw matrix is
therefore almost entirely a translation loss; the projective terms contribute nothing to the
gradient and never get learned.

So we standardise per element. Two further choices matter:

- **`M[2,2]` is fixed to 1.** A homography is only defined up to scale, so leaving all nine
  values free gives the network a redundant degree of freedom — and lets it predict a matrix
  with `M[2,2] ≈ 0`, which is singular and makes `torch.inverse` explode. The network
  predicts the **8** remaining parameters.
- **The network predicts in normalised space.** Its output is therefore roughly `N(0,1)`,
  and a freshly initialised network (output ≈ 0) predicts the *dataset mean* homography —
  a sane, invertible starting guess rather than a near-zero singular matrix.

In [ ]:
def compute_matrix_stats(matrix_dir):
    """Per-element mean and std over all matrices, after scaling each so M[2,2] = 1."""
    files = sorted(f for f in os.listdir(matrix_dir) if f.endswith('.npy'))
    mats = []
    for f in files:
        m = np.load(os.path.join(matrix_dir, f)).astype(np.float32)
        mats.append(m / (m[2, 2] + 1e-8))
    mats = np.stack(mats)
    mean = mats.mean(axis=0)
    std = np.maximum(mats.std(axis=0), 1e-6)
    return torch.from_numpy(mean), torch.from_numpy(std)


def matrix_to_vec(matrix, mean, std):
    """(B, 3, 3) raw homography -> (B, 8) normalised free parameters."""
    mean, std = mean.to(matrix.device), std.to(matrix.device)
    scaled = matrix / (matrix[:, 2:3, 2:3] + 1e-8)
    normed = (scaled - mean) / std
    return normed.reshape(-1, 9)[:, :8]


def vec_to_matrix(vec, mean, std):
    """(B, 8) normalised parameters -> (B, 3, 3) raw homography with M[2,2] = 1."""
    mean, std = mean.to(vec.device), std.to(vec.device)
    pad = torch.zeros(vec.size(0), 1, device=vec.device, dtype=vec.dtype)
    full = torch.cat([vec, pad], dim=1).view(-1, 3, 3)
    matrix = full * std + mean
    # force the bottom-right element to exactly 1 without an in-place write
    keep = torch.ones(3, 3, device=vec.device, dtype=vec.dtype)
    keep[2, 2] = 0.0
    bias = torch.zeros(3, 3, device=vec.device, dtype=vec.dtype)
    bias[2, 2] = 1.0
    return matrix * keep + bias


def perspective_ocr_collate_fn(batch):
    batch = [b for b in batch if b is not None]
    images = torch.stack([b['image'] for b in batch], 0)
    matrices = torch.stack([b['matrix'] for b in batch], 0)
    orig_sizes = torch.stack([b['orig_size'] for b in batch], 0)
    labels = pad_sequence([b['label'] for b in batch], batch_first=True, padding_value=0)
    return images, matrices, labels, orig_sizes

## Load dataset

In [ ]:
USE_COLAB = True

if USE_COLAB:
    # Google Drive file id of the dataset zip (the part of the share link after /d/).
    # Use the pre-resized zip from data_generation/prepare_training_set.py if you have it.
    DATA_ZIP_ID = ""
    if not DATA_ZIP_ID:
        raise ValueError("Set DATA_ZIP_ID to the Drive file id of your dataset zip.")
    !pip install -q gdown
    !gdown -q {DATA_ZIP_ID} -O data.zip
    !unzip -q -o data.zip -d /content/
    ROOT = "/content/perspective_dataset_256"      # or perspective_dataset_with_cards
else:
    ROOT = "perspective_dataset_with_cards"

if not os.path.isdir(ROOT):
    raise FileNotFoundError(
        f"{ROOT} not found. Check what the zip actually extracted to: "
        "run `!ls /content` and set ROOT to match.")

IMG_DIR = f"{ROOT}/images"
MAT_DIR = f"{ROOT}/matrices"
LBL_DIR = f"{ROOT}/labels"

dataset = PerspectiveOCRDataset(IMG_DIR, MAT_DIR, LBL_DIR, img_size=IMG_SIZE,
                                orig_sizes=f"{ROOT}/orig_sizes.json")
matrix_mean, matrix_std = compute_matrix_stats(MAT_DIR)

sample = dataset[0]
print(f"Dataset size : {len(dataset)}")
print(f"Image        : {tuple(sample['image'].shape)}")
print(f"Original size: {sample['orig_size'].tolist()}")
print(f"Label        : {sample['label'].tolist()}")
print(f"\nMatrix mean:\n{matrix_mean}")
print(f"\nMatrix std:\n{matrix_std}")

## Model

In [ ]:
class PerspectiveNet(nn.Module):
    """Predicts the 8 free parameters of the homography, in normalised space."""

    def __init__(self, in_channels=3):
        super().__init__()

        def block(cin, cout, pool=True):
            layers = [nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
            if pool:
                layers.append(nn.MaxPool2d(2, 2))
            return layers

        self.features = nn.Sequential(
            *block(in_channels, 32),   # 256 -> 128
            *block(32, 64),            # 128 -> 64
            *block(64, 128),           # 64  -> 32
            *block(128, 256),          # 32  -> 16
            *block(256, 512),          # 16  -> 8
            *block(512, 512, pool=False),
            nn.AdaptiveAvgPool2d((4, 4)),
        )

        self.regressor = nn.Sequential(
            nn.Linear(512 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(1024, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 8),
        )

        # start at the dataset-mean homography: zeros in normalised space
        nn.init.zeros_(self.regressor[-1].weight)
        nn.init.zeros_(self.regressor[-1].bias)

    def forward(self, x):
        x = self.features(x)
        return self.regressor(x.flatten(1))   # (B, 8) normalised

In [ ]:
class DifferentiableSpatialTransformer(nn.Module):
    """
    Straightens the card by inverse-warping with the predicted homography.

    The matrix maps ORIGINAL-resolution pixels into a RECT_W x RECT_H rectangle, so the
    destination grid must span that rectangle and the resulting source coordinates must be
    normalised by the ORIGINAL image size — not by the size of the resized tensor we sample
    from. grid_sample's [-1, 1] coordinates are resolution independent, which is what makes
    sampling the resized image valid.
    """

    def __init__(self, output_size=CARD_SIZE, rect_size=(RECT_W, RECT_H)):
        super().__init__()
        self.out_h, self.out_w = output_size
        self.rect_w, self.rect_h = rect_size

    def forward(self, image, matrix, orig_size):
        B = image.size(0)
        device = image.device

        # destination grid, spanning the rectangle the matrix maps into
        ys, xs = torch.meshgrid(
            torch.linspace(0, self.rect_h - 1, self.out_h, device=device),
            torch.linspace(0, self.rect_w - 1, self.out_w, device=device),
            indexing='ij')
        grid = torch.stack([xs.flatten(), ys.flatten(), torch.ones(xs.numel(), device=device)])
        grid = grid.unsqueeze(0).expand(B, -1, -1)                      # (B, 3, N)

        # inverse-map into original-resolution source pixels
        src = torch.linalg.solve(matrix, grid)                          # (B, 3, N)
        denom = src[:, 2, :]
        denom = torch.where(denom.abs() < 1e-8, torch.full_like(denom, 1e-8), denom)
        src_x, src_y = src[:, 0, :] / denom, src[:, 1, :] / denom

        # normalise by the ORIGINAL image size
        w = orig_size[:, 0].unsqueeze(1)
        h = orig_size[:, 1].unsqueeze(1)
        norm = torch.stack([2.0 * src_x / (w - 1) - 1.0,
                            2.0 * src_y / (h - 1) - 1.0], dim=-1)
        norm = norm.view(B, self.out_h, self.out_w, 2)

        return F.grid_sample(image, norm, mode='bilinear',
                             padding_mode='zeros', align_corners=True)

In [ ]:
class OCRNet(nn.Module):
    """
    Reads 16 digits from a (B, 3, 64, 128) straightened crop.

    The last two pools are height-only. Pooling width five times would leave a 4-wide feature
    map, and adaptively pooling that up to 16 positions would hand four consecutive digits
    identical features — capping accuracy no matter how long you train. Keeping width at 16
    gives each digit position its own column.
    """

    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                    # 64x128 -> 32x64

            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                    # -> 16x32

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                                    # -> 8x16

            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1)),                                  # -> 4x16  (width kept)

            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1)),                                  # -> 2x16  (width kept)

            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 16)),                         # -> 1x16
        )
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x).squeeze(2)   # (B, 512, 16)
        return self.classifier(x.permute(0, 2, 1))   # (B, 16, 10)

In [ ]:
class PerspectiveOCRModel(nn.Module):
    def __init__(self, matrix_mean, matrix_std, card_size=CARD_SIZE):
        super().__init__()
        self.perspective_net = PerspectiveNet(in_channels=3)
        self.spatial_transformer = DifferentiableSpatialTransformer(output_size=card_size)
        self.ocr_net = OCRNet(in_channels=3, num_classes=10)
        # buffers so they move with .to(device) and get saved in the state dict
        self.register_buffer('matrix_mean', matrix_mean)
        self.register_buffer('matrix_std', matrix_std)

    def forward(self, x, orig_size):
        pred_vec = self.perspective_net(x)                                   # (B, 8) normalised
        pred_matrix = vec_to_matrix(pred_vec, self.matrix_mean, self.matrix_std)
        straightened = self.spatial_transformer(x, pred_matrix, orig_size)
        digit_logits = self.ocr_net(straightened)
        return pred_vec, pred_matrix, straightened, digit_logits

## Sanity check the transformer before training

Warp with the **ground-truth** matrices. If this does not produce readable cards, nothing
downstream can work, and it is far cheaper to find out here than after 50 epochs.

In [ ]:
def show_gt_straightening(dataset, n=4):
    st = DifferentiableSpatialTransformer()
    fig, axes = plt.subplots(n, 2, figsize=(10, 2.4 * n))
    denorm = lambda t: (t * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

    for row in range(n):
        s = dataset[row * (len(dataset) // n)]
        with torch.no_grad():
            out = st(s['image'].unsqueeze(0), s['matrix'].unsqueeze(0), s['orig_size'].unsqueeze(0))
        digits = ''.join(str(d) for d in s['label'].tolist())
        axes[row, 0].imshow(denorm(s['image']));  axes[row, 0].set_title('input'); axes[row, 0].axis('off')
        axes[row, 1].imshow(denorm(out[0]))
        axes[row, 1].set_title(' '.join(digits[i:i+4] for i in range(0, 16, 4)))
        axes[row, 1].axis('off')

    plt.tight_layout(); plt.show()


show_gt_straightening(dataset)

## Training

In [ ]:
def run_epoch(model, loader, mse_criterion, ce_criterion, alpha, beta, device, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    totals = np.zeros(3)

    with torch.set_grad_enabled(train):
        for images, gt_matrices, gt_labels, orig_sizes in loader:
            images, gt_matrices = images.to(device), gt_matrices.to(device)
            gt_labels, orig_sizes = gt_labels.to(device), orig_sizes.to(device)

            pred_vec, _, _, digit_logits = model(images, orig_sizes)

            gt_vec = matrix_to_vec(gt_matrices, model.matrix_mean, model.matrix_std)
            loss_mse = mse_criterion(pred_vec, gt_vec)

            nd = min(digit_logits.size(1), gt_labels.size(1))
            loss_ce = ce_criterion(digit_logits[:, :nd, :].reshape(-1, 10),
                                   gt_labels[:, :nd].reshape(-1))

            loss = alpha * loss_mse + beta * loss_ce

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

            totals += np.array([loss.item(), loss_mse.item(), loss_ce.item()])

    return totals / len(loader)


def train_model(model, train_loader, val_loader, num_epochs=50, lr=1e-4,
                device='cuda', alpha=1.0, beta=1.0):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    mse_criterion, ce_criterion = nn.MSELoss(), nn.CrossEntropyLoss()
    history = {k: [] for k in ('train_loss', 'val_loss', 'train_mse', 'val_mse', 'train_ce', 'val_ce')}

    print(f"Training on {device} | alpha={alpha}, beta={beta}")
    print(f"{'Epoch':>6} {'Train':>10} {'Val':>10} {'Val MSE':>10} {'Val CE':>10}")
    print("-" * 50)

    for epoch in range(num_epochs):
        tr = run_epoch(model, train_loader, mse_criterion, ce_criterion, alpha, beta, device, optimizer)
        va = run_epoch(model, val_loader, mse_criterion, ce_criterion, alpha, beta, device)
        scheduler.step(va[0])

        for key, val in zip(('train_loss', 'train_mse', 'train_ce'), tr): history[key].append(val)
        for key, val in zip(('val_loss', 'val_mse', 'val_ce'), va): history[key].append(val)

        if epoch == 0 or (epoch + 1) % 5 == 0:
            print(f"{epoch+1:>6} {tr[0]:>10.4f} {va[0]:>10.4f} {va[1]:>10.4f} {va[2]:>10.4f}")

    return model, history

In [ ]:
BATCH_SIZE, LEARNING_RATE, EPOCHS = 16, 1e-4, 50
ALPHA, BETA = 1.0, 1.0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(42)
generator = torch.Generator().manual_seed(42)
train_size = int(0.8 * len(dataset))
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, len(dataset) - train_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=perspective_ocr_collate_fn,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=perspective_ocr_collate_fn,
                        num_workers=2, pin_memory=True)

model = PerspectiveOCRModel(matrix_mean, matrix_std)
print(f"Train: {train_size}, Val: {len(dataset) - train_size}, Device: {DEVICE}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

model, history = train_model(model, train_loader, val_loader, num_epochs=EPOCHS,
                             lr=LEARNING_RATE, device=DEVICE, alpha=ALPHA, beta=BETA)

In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (key, title) in zip(axes, [('loss', 'Total Loss'),
                                       ('mse', 'Matrix MSE (normalised)'),
                                       ('ce', 'OCR CrossEntropy')]):
        ax.plot(history[f'train_{key}'], label='Train')
        ax.plot(history[f'val_{key}'], label='Val')
        ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)
    plt.tight_layout(); plt.show()


plot_history(history)

## Evaluation

In [ ]:
def evaluate_accuracy(model, loader, device, num_digits=16):
    """Strict sequence accuracy (all 16 digits right), plus per-digit accuracy."""
    model.eval().to(device)
    correct_seq = correct_dig = total_dig = total = 0
    total_mse = 0.0
    mse_criterion = nn.MSELoss()

    with torch.no_grad():
        for images, gt_matrices, gt_labels, orig_sizes in loader:
            images, gt_matrices = images.to(device), gt_matrices.to(device)
            gt_labels, orig_sizes = gt_labels.to(device), orig_sizes.to(device)

            pred_vec, _, _, digit_logits = model(images, orig_sizes)
            gt_vec = matrix_to_vec(gt_matrices, model.matrix_mean, model.matrix_std)
            total_mse += mse_criterion(pred_vec, gt_vec).item()

            nd = min(digit_logits.size(1), gt_labels.size(1), num_digits)
            matches = digit_logits[:, :nd, :].argmax(dim=2).eq(gt_labels[:, :nd])

            correct_seq += matches.all(dim=1).sum().item()
            correct_dig += matches.sum().item()
            total_dig += matches.numel()
            total += images.size(0)

    seq_acc, dig_acc = 100.0 * correct_seq / total, 100.0 * correct_dig / total_dig
    print("=" * 44)
    print(f"Samples             : {total}")
    print(f"Sequence accuracy   : {seq_acc:.2f}%  ({correct_seq}/{total})")
    print(f"Per-digit accuracy  : {dig_acc:.2f}%")
    print(f"Matrix MSE (normed) : {total_mse / len(loader):.6f}")
    print("=" * 44)
    return seq_acc, dig_acc, total_mse / len(loader)


evaluate_accuracy(model, val_loader, DEVICE)

## Predictions

Input, the crop produced by the ground-truth matrix, and the crop produced by the predicted
matrix. Comparing the last two columns shows whether geometry or reading is the bottleneck.

In [ ]:
def visualize_predictions(model, dataset, device, num_samples=6, seed=0):
    model.eval().to(device)
    gt_st = DifferentiableSpatialTransformer().to(device)
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset), num_samples, replace=False)

    fig, axes = plt.subplots(num_samples, 3, figsize=(13, 2.6 * num_samples))
    denorm = lambda t: (t * 0.5 + 0.5).clamp(0, 1).cpu().permute(1, 2, 0).numpy()
    group = lambda s: ' '.join(s[i:i+4] for i in range(0, len(s), 4))

    for row, idx in enumerate(indices):
        s = dataset[int(idx)]
        image = s['image'].unsqueeze(0).to(device)
        orig = s['orig_size'].unsqueeze(0).to(device)

        with torch.no_grad():
            _, _, pred_straight, digit_logits = model(image, orig)
            gt_straight = gt_st(image, s['matrix'].unsqueeze(0).to(device), orig)

        pred_text = group(''.join(str(d) for d in digit_logits.argmax(dim=2)[0].cpu().tolist()))
        gt_text = group(''.join(str(d) for d in s['label'].tolist()))
        ok = '✓' if pred_text == gt_text else '✗'

        for col, (img, title) in enumerate([
                (denorm(s['image']), 'input'),
                (denorm(gt_straight[0]), f'GT matrix\n{gt_text}'),
                (denorm(pred_straight[0]), f'predicted {ok}\n{pred_text}')]):
            axes[row, col].imshow(img); axes[row, col].set_title(title, fontsize=9)
            axes[row, col].axis('off')

    plt.tight_layout(); plt.show()


visualize_predictions(model, dataset, DEVICE)

## Save

In [ ]:
torch.save({'state_dict': model.state_dict(),
            'matrix_mean': matrix_mean,
            'matrix_std': matrix_std}, 'perspective_ocr_model.pth')
print('Saved to perspective_ocr_model.pth')